In [1]:
# After running, restart the kernel before continuing.
%pip -q install --upgrade "sagemaker>=2,<3" boto3 botocore mlflow
%pip -q install --upgrade "imbalanced-learn>=0.11"
%pip -q install --upgrade "xgboost>=2.0,<3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
aiobotocore 3.8.0 requires botocore<1.43.47,>=1.43.3, but you have botocore 1.43.78 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.6.0 which is incompatible.
sagemaker-mlops 1.12.0 requires sagemaker-core>=2.12.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.12.0 requires sagemaker-core>=2

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [1]:
# Standard Library Imports
import os

# 1. Suppress the SageMaker v2 deprecation warning
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

# Standard Library Imports
import json
import shutil
import time

from pathlib import Path
from urllib.parse import urlparse

# Data & ML Libraries
import imblearn
import sklearn
import xgboost

from sklearn.model_selection import train_test_split

# Cloud & MLOps
import boto3
import mlflow

from botocore.exceptions import ClientError

# AWS SageMaker SDK
import sagemaker

from sagemaker import ModelPackage
from sagemaker.model import Model
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.serverless import ServerlessInferenceConfig

from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.sklearn.processing import SKLearnProcessor

from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import ProcessingStep, TrainingStep

print(f"Imbalanced-learn Version: {imblearn.__version__}")
print(f"SageMaker Version: {sagemaker.__version__}")
print(f"Scikit-learn Version: {sklearn.__version__}")
print(f"XGboost Version: {xgboost.__version__}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Imbalanced-learn Version: 0.14.2
SageMaker Version: 2.257.6
Scikit-learn Version: 1.7.2
XGboost Version: 2.1.4


In [2]:
# ----------------------------
# AWS / SageMaker setup
# ----------------------------
session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

# Use the ITI113 course bucket and team prefix.
# This matches the team execution role S3 policy, e.g.:
# s3://nyp-26s1-iti113/iti113/team40/
BUCKET  = "nyp-26s1-iti113"

# Change these for the current student/profile.
TEAM_ID = "team09"
STUDENT_ID = "s901"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "bank-fraud-detection"

PREFIX  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

# Instance types.
# If ml.m5.large quota is 0, change these to an approved available training/processing type.
# For ITI113, keep Studio spaces on ml.t3.medium and use SageMaker jobs for training/processing.
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"

# ----------------------------
# SageMaker Serverless MLflow App setup
# ----------------------------
# Preferred:
# 1. Team-level config copied/shared from Notebook 01A:
#       mlflow_app_config_team40.json
# 2. Student-specific config from Notebook 01A:
#       mlflow_app_config_team40_s4002.json
# 3. Any local config matching this team:
#       mlflow_app_config_team40_*.json
#
# Important:
# Do not use another team's MLflow ARN. With team-level IAM
# restriction, wrong-team access should fail with 403.
# ----------------------------
TEAM_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}.json")
STUDENT_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")

config_candidates = [
    TEAM_CONFIG_FILE,
    STUDENT_CONFIG_FILE,
    *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json")),
]

MLFLOW_APP_ARN = None
MLFLOW_EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment2"
mlflow_config = {}
config_used = None

for config_file in config_candidates:
    if config_file.exists():
        mlflow_config = json.loads(config_file.read_text(encoding="utf-8"))
        MLFLOW_APP_ARN = (
            mlflow_config.get("MLFLOW_APP_ARN")
            or mlflow_config.get("mlflow_app_arn")
            or mlflow_config.get("arn")
        )
        MLFLOW_EXPERIMENT_NAME = (
            mlflow_config.get("EXPERIMENT_NAME")
            or mlflow_config.get("experiment_name")
            or MLFLOW_EXPERIMENT_NAME
        )
        config_used = config_file
        break

# Fallback for classroom testing only.
# Update this to your team's MLflow App ARN from Notebook 01A if the config file
# is not available in this Studio workspace.
DEFAULT_MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:"
    "mlflow-app/app-L5IMA5YSBDTY"
)

if MLFLOW_APP_ARN is None:
    MLFLOW_APP_ARN = DEFAULT_MLFLOW_APP_ARN
    print(
        "[WARNING] No local MLflow config file found. "
        "Using DEFAULT_MLFLOW_APP_ARN. Make sure this ARN belongs to your own team."
    )
else:
    print(f"Loaded MLflow App config from {config_used}")

# Validate config team, if present.
config_team_id = mlflow_config.get("TEAM_ID") or mlflow_config.get("team_id")
if config_team_id and config_team_id != TEAM_ID:
    raise ValueError(
        f"Config file team mismatch: config TEAM_ID={config_team_id}, notebook TEAM_ID={TEAM_ID}. "
        "Do not use another team's MLflow config."
    )
# ----------------------------
# Safety check for team-level MLflow restriction
# ----------------------------
# The selected MLflow App must have ResourceTag/TeamId = TEAM_ID.
# If a student accidentally uses another team's ARN, this should either:
# - fail with AccessDenied / 403 due to IAM restriction, or
# - fail this explicit validation before logging.
# ----------------------------

sm_for_mlflow = boto3.client("sagemaker", region_name=region)

try:
    tag_response = sm_for_mlflow.list_tags(ResourceArn=MLFLOW_APP_ARN)
    mlflow_app_tags = {t["Key"]: t["Value"] for t in tag_response.get("Tags", [])}

    print("MLflow App tags:")
    for k, v in mlflow_app_tags.items():
        print(f"  {k}: {v}")

    app_team_id = mlflow_app_tags.get("TeamId")
    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag mismatch. App TeamId={app_team_id}, notebook TEAM_ID={TEAM_ID}. "
            "Do not log to another team's MLflow App."
        )

    print(f"[OK] MLflow App tag TeamId={app_team_id} matches notebook TEAM_ID={TEAM_ID}")

except Exception as e:
    print("\n[ERROR] Could not validate MLflow App team tag.")
    print("This usually means one of the following:")
    print("1. The MLflow App ARN belongs to another team and IAM correctly blocked access.")
    print("2. The MLflow App is missing the TeamId tag.")
    print("3. The current role lacks permission to list tags for this MLflow App.")
    print(type(e).__name__, e)
    raise

# Optional: store for downstream cells and subprocesses.
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_APP_ARN
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
# ----------------------------
# MLflow App UI link helpers
# ----------------------------
# MLflow may print generic links such as:
# https://mlflow.sagemaker.ap-southeast-1.app.aws/#/...
# Those links are not presigned and may show a permission/session error.
# Use these helpers to generate a fresh presigned SageMaker MLflow App URL
# and append the experiment/run fragment.

def create_mlflow_app_presigned_url(fragment: str = "") -> str:
    sm_for_mlflow = boto3.client("sagemaker", region_name=region)
    response = sm_for_mlflow.create_presigned_mlflow_app_url(
        Arn=MLFLOW_APP_ARN
    )

    base_url = response.get("AuthorizedUrl") or response.get("Url")

    if not base_url:
        raise RuntimeError(
            "create_presigned_mlflow_app_url did not return AuthorizedUrl or Url. "
            f"Response: {response}"
        )

    # Remove any existing fragment before appending our own MLflow UI route.
    base_url = base_url.split("#", 1)[0]

    if fragment:
        return base_url + "#" + fragment.lstrip("#")

    return base_url


def print_mlflow_presigned_links(experiment_id=None, run_id=None):
    if experiment_id is not None:
        experiment_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}"
        )
        print("Presigned MLflow experiment URL:")
        print(experiment_url)

    if experiment_id is not None and run_id is not None:
        run_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}/runs/{run_id}"
        )
        print("\nPresigned MLflow run URL:")
        print(run_url)

PIPELINE_NAME       = f"iti113-{TEAM_ID}-bank-fraud-detection"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-BankFraudDetection"
ENDPOINT_NAME       = f"iti113-{TEAM_ID}-bank-fraud-detection"
QUALITY_GATE_AUC    = 0.60

RAW_DATA_URI  = f"s3://{BUCKET}/{PREFIX}/raw/bank_fraud.csv"
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline"

# Store the pipeline source files in S3 first, then download them into a clean local folder.
SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI    = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

print(f"Pipeline                : {PIPELINE_NAME}")
print(f"Bucket                  : {BUCKET}")
print(f"Team prefix             : {PREFIX}")
print(f"Semester                : {SEMESTER}")
print(f"Region                  : {region}")
print(f"SageMaker role          : {role}")
print(f"MLflow App ARN          : {MLFLOW_APP_ARN}")
print(f"MLflow experiment       : {MLFLOW_EXPERIMENT_NAME}")
print(f"Pipeline source S3 URI  : {SCRIPTS_S3_URI}")
print(f"Local pipeline source   : {LOCAL_PIPELINE_SRC}")

Loaded MLflow App config from mlflow_app_config_team09_s901.json


MLflow App tags:
  Semester: 26S1
  sagemaker:domain-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-8nb3rzhhmygx
  ProjectName: bank-fraud-detection
  sagemaker:space-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-8nb3rzhhmygx/team09-shared
  Course: ITI113
  TeamId: team09
  CreatedByNotebook: 01A_fraud_setup_sagemaker_mlflow_app
  StudentId: s901
[OK] MLflow App tag TeamId=team09 matches notebook TEAM_ID=team09
Pipeline                : iti113-team09-bank-fraud-detection
Bucket                  : nyp-26s1-iti113
Team prefix             : iti113/team09/data/bank-fraud-detection
Semester                : 26S1
Region                  : ap-southeast-1
SageMaker role          : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team09
MLflow App ARN          : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-YLZFIPQTQXFU
MLflow experiment       : ITI113/team09/Experiment1
Pipeline source S3 URI  : s3://nyp-26s1-iti113/iti113/team09/data/bank-

In [3]:
# Mlflow precheck and setup presigned link
mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"{TEAM_ID}_pipeline_notebook_precheck_{int(time.time())}") as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "bank_fraud",
        "run_type": "sagemaker_pipeline_precheck",
        "tracking_backend": "sagemaker_mlflow_app",
        "mlflow_app_arn": MLFLOW_APP_ARN,
    })
    mlflow.log_param("source", "notebook_03_precheck")
    mlflow.log_metric("connection_success", 1)

    precheck_run_id = run.info.run_id
    precheck_experiment_id = run.info.experiment_id

print("SageMaker MLflow App precheck completed.")
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", precheck_experiment_id)
print("Run ID:", precheck_run_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=precheck_experiment_id,
    run_id=precheck_run_id
)


🏃 View run team09_pipeline_notebook_precheck_1787413684 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/808d3b4fa0be41beb39be6fc0e988add
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
SageMaker MLflow App precheck completed.
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-YLZFIPQTQXFU
Experiment: ITI113/team09/Experiment1
Experiment ID: 1
Run ID: 808d3b4fa0be41beb39be6fc0e988add

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-YLZFIPQTQXFU.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlpXSTNHTSIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHVab3d2RENOS3JteSttQXBRK25sSm1sM1ROcTlNNHNubmZuRS9nTE9qTTRBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFcllUbHlOMnA0T1hKb1V6Tkhka2MyUmtrM1lqRXphbTEyVW5KNEwydEtMM1pOYUhaemFEWndkWFV5VVZNelEyRjJkRmgyTTFKcVREVTBRV2hSWld0aVp6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFaMlJyc3JGTys3SmozWmZKdlUzZk9nQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6Z0o1OXZxVVNWS054bjRoTUNBUkNBTzc3ZnQ4dnREMDdYYWx1T044dTUyQTRyY2hOQk5YMzhKYzhUVmNiT2xMUFpFQzB6bTdDYkQzQjIxektydmEwTE9VVFgyTkdTdjVDS1N5Mm5BZ0FBRUFBWlhTVXFsZjBtTTdnZ291NC9keEszK2JsRHpmTHJzOWhXUnZLWWdXaFR3cUsrS1hvanJpck


Presigned MLflow run URL:
https://app-YLZFIPQTQXFU.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IjZNMldNVCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHh0cVByN1JUajN3UjkyM0ZPMHRiOXluU0VVbFo5RTNvMkRwRVJCMEMvd0lBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFcmQycFVXQzk1UkZaTk5rRlJlR2xqUkZrMlFtbEljVEJ0YVRORlozZE5ibkp1Y0hOYWFtSkRhRGhsUkhCWGNHSTRTbWxHZWxGV01tVlZPRkpqWjI1WFVUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFiWERQanpqQ2JLaUtlOHRsdVF1cVZrQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6T2FjMEJxRk1idWN5TS9lSUNBUkNBTzhNY1Fpa1Z6UXNMaUIramhwUzNvQk9ISVR6Q3hYcFQ5K1dLbXVjVmlYRHN3REVOZkNEeXdrcldPaTNWbHFCUmxHNWFEYzFMVDMyNTlYNEpBZ0FBRUFEeDMxbGR0VW9WNGhzcnh3L0gvTnIyZXc1TFUzZDY1SWNqWjdkeWxVTE1xSHNEdk1HeStPWnBIZmVW

In [4]:
# Setup local src folder
os.makedirs("src", exist_ok=True)

with open("src/requirements.txt", "w") as f:
    f.write("xgboost>=2.0,<3\n")
    f.write("imbalanced-learn>=0.11\n")
    f.write("scikit-learn>=1.2.0\n")

print("src/ directory ready")
print("src/requirements.txt written")

src/ directory ready
src/requirements.txt written


**Insight — `preprocess.py` re-implements the exact 32-numeric/14-categorical feature-engineering logic from Notebook 01**, so a SageMaker Processing job run through the pipeline produces byte-for-byte the same engineered features as the interactive notebooks. This duplication (rather than a shared importable module) is a known maintenance risk: if Notebook 01's logic is ever updated, this script must be manually kept in sync.

In [65]:
%%writefile src/preprocess.py
import argparse
import logging
import os

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

TARGET_COLUMN = "is_fraud"


class PreprocessedModel(BaseEstimator):
    """Wrap a fitted preprocessor + classifier so joblib can serialize it
    and inference.py can call predict_proba(raw features) directly.
    """

    def __init__(self, prep, clf):
        self.prep = prep
        self.clf = clf

    def predict_proba(self, X):
        return self.clf.predict_proba(self.prep.transform(X))

    @property
    def named_steps(self):
        return {"prep": self.prep, "clf": self.clf}


NUMERIC_FEATURES = [
    "hour_of_day",
    "customer_age",
    "credit_score",
    "account_age_years",
    "account_balance",
    "transaction_amount",
    "num_prev_transactions",
    "transaction_freq_monthly",
    "distance_from_home_km",
    "time_since_last_txn_hrs",
    "failed_attempts",
    "log_transaction_amount",
    "log_account_balance",
    "amount_to_balance_ratio",
    "risk_score",
    "failed_x_transaction",
    "night_x_international",
    "pin_x_failed",
    "night_x_pin",
    "amount_x_distance",
    "amount_x_failed",
    "risk_x_amount",
    "risk_x_distance",
    "customer_mean_amount",
    "customer_std_amount",
    "customer_mean_distance",
    "customer_std_distance",
    "customer_txn_count",
    "customer_mean_balance",
    "amount_zscore",
    "distance_zscore",
    "balance_deviation",
]
CATEGORICAL_FEATURES = [
    "is_weekend",
    "is_night_transaction",
    "country",
    "city",
    "merchant_category",
    "payment_method",
    "device_type",
    "is_international",
    "pin_changed_recently",
    "customer_age_group",
    "hour_bin",
    "high_amount",
    "far_from_home",
    "rapid_txn",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

DROP_COLS = [
    "transaction_id",
    "customer_id",
    "transaction_date",
    "transaction_time",
    "fraud_type",
]

# Defaults for raw transaction fields, used when a caller omits columns at inference
RAW_DEFAULTS = {
    "transaction_id": "TXN9990000001",
    "customer_id": "CUST99000000",
    "transaction_date": "2024-01-01",
    "transaction_time": "12:00:00",
    "hour_of_day": 12,
    "is_weekend": 0,
    "is_night_transaction": 0,
    "country": "USA",
    "city": "New York",
    "merchant_category": "Grocery",
    "payment_method": "Debit Card",
    "device_type": "Mobile",
    "customer_age": 35,
    "credit_score": 600,
    "account_age_years": 5.0,
    "account_balance": 1000.0,
    "transaction_amount": 100.0,
    "num_prev_transactions": 10,
    "transaction_freq_monthly": 5,
    "distance_from_home_km": 0.0,
    "time_since_last_txn_hrs": 24.0,
    "is_international": 0,
    "failed_attempts": 0,
    "pin_changed_recently": 0,
    "is_fraud": 0,
    "fraud_type": "",
}

# The exact column order of the original bank_fraud.csv, used for headerless CSV inference
RAW_COLUMNS = [
    'transaction_id',
    'customer_id',
    'transaction_date',
    'transaction_time',
    'hour_of_day',
    'is_weekend',
    'is_night_transaction',
    'country',
    'city',
    'merchant_category',
    'payment_method',
    'device_type',
    'customer_age',
    'credit_score',
    'account_age_years',
    'account_balance',
    'transaction_amount',
    'num_prev_transactions',
    'transaction_freq_monthly',
    'distance_from_home_km',
    'time_since_last_txn_hrs',
    'is_international',
    'failed_attempts',
    'pin_changed_recently',
    'is_fraud',
    'fraud_type'
]


def _get_series(df, col, default):
    """Return a column as a Series, or a constant Series if the column is missing."""
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


def _to_int(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).astype(int)


def _to_float(series):
    return pd.to_numeric(series, errors="coerce").fillna(0.0)


def engineer_features(df, sort=True, split_year=2022):
    df = df.copy()

    # Build a single datetime column from date and time strings
    if "transaction_datetime" not in df.columns:
        date_series = _get_series(df, "transaction_date", RAW_DEFAULTS["transaction_date"]).astype(str)
        time_series = _get_series(df, "transaction_time", RAW_DEFAULTS["transaction_time"]).astype(str)
        df["transaction_datetime"] = pd.to_datetime(date_series + " " + time_series, errors="coerce")

    # Sorting is only useful during training preprocessing. During real-time
    # inference it would reorder batch requests and misalign predictions with
    # the original input order, so prepare_transactions() calls with sort=False
    if sort:
        df = df.sort_values("transaction_datetime").reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    # Extract year BEFORE dropping columns or returning
    df["year"] = df["transaction_datetime"].dt.year

    # Numeric Feature Engineering
    transaction_amount = _to_float(_get_series(df, "transaction_amount", RAW_DEFAULTS["transaction_amount"]))
    account_balance = _to_float(_get_series(df, "account_balance", RAW_DEFAULTS["account_balance"]))
    distance_from_home_km = _to_float(_get_series(df, "distance_from_home_km", RAW_DEFAULTS["distance_from_home_km"]))
    time_since_last_txn_hrs = _to_float(_get_series(df, "time_since_last_txn_hrs", RAW_DEFAULTS["time_since_last_txn_hrs"]))
    transaction_freq_monthly = _to_float(_get_series(df, "transaction_freq_monthly", RAW_DEFAULTS["transaction_freq_monthly"]))
    account_age_years = _to_float(_get_series(df, "account_age_years", RAW_DEFAULTS["account_age_years"]))
    num_prev_transactions = _to_float(_get_series(df, "num_prev_transactions", RAW_DEFAULTS["num_prev_transactions"]))

    df["log_transaction_amount"] = np.log1p(transaction_amount)
    df["log_account_balance"] = np.log1p(account_balance)
    df["amount_to_balance_ratio"] = transaction_amount / (account_balance + 1.0)

    # Behavioural flags
    df["is_night_transaction"] = _to_int(_get_series(df, "is_night_transaction", RAW_DEFAULTS["is_night_transaction"]))
    df["is_international"] = _to_int(_get_series(df, "is_international", RAW_DEFAULTS["is_international"]))
    df["failed_attempts"] = _to_int(_get_series(df, "failed_attempts", RAW_DEFAULTS["failed_attempts"]))
    df["pin_changed_recently"] = _to_int(_get_series(df, "pin_changed_recently", RAW_DEFAULTS["pin_changed_recently"]))
    df["is_weekend"] = _to_int(_get_series(df, "is_weekend", RAW_DEFAULTS["is_weekend"]))

    df["risk_score"] = (
        df["is_night_transaction"].astype(int)
        + df["is_international"].astype(int)
        + (df["failed_attempts"] > 0).astype(int)
        + df["pin_changed_recently"].astype(int)
    )

    customer_age = _to_float(_get_series(df, "customer_age", RAW_DEFAULTS["customer_age"]))
    df["customer_age_group"] = pd.cut(
        customer_age,
        bins=[0, 25, 40, 60, 100],
        labels=["18-25", "26-40", "41-60", "60+"],
    ).astype(str)

    hour_of_day = _to_float(_get_series(df, "hour_of_day", RAW_DEFAULTS["hour_of_day"]))
    df["hour_bin"] = pd.cut(
        hour_of_day,
        bins=[-1, 5, 11, 17, 23],
        labels=["night_0_5", "morning_6_11", "afternoon_12_17", "evening_18_23"],
    ).astype(str)

    df["failed_x_transaction"] = df["failed_attempts"] * df["is_international"]
    df["night_x_international"] = df["is_night_transaction"] * df["is_international"]
    df["pin_x_failed"] = df["pin_changed_recently"] * df["failed_attempts"]
    df["night_x_pin"] = df["is_night_transaction"] * df["pin_changed_recently"]
    df["amount_x_distance"] = df["log_transaction_amount"] * df["distance_from_home_km"]
    df["amount_x_failed"] = df["log_transaction_amount"] * df["failed_attempts"]
    df["risk_x_amount"] = df["risk_score"] * df["log_transaction_amount"]
    df["risk_x_distance"] = df["risk_score"] * df["distance_from_home_km"]

    train_period = df["transaction_datetime"].dt.year <= split_year
    if train_period.any():
        amount_p90 = df.loc[train_period, "transaction_amount"].quantile(0.90) if "transaction_amount" in df.columns else transaction_amount[train_period].quantile(0.90)
        distance_p90 = distance_from_home_km[train_period].quantile(0.90)
        time_p10 = time_since_last_txn_hrs[train_period].quantile(0.10)
    else:
        amount_p90 = transaction_amount.quantile(0.90)
        distance_p90 = distance_from_home_km.quantile(0.90)
        time_p10 = time_since_last_txn_hrs.quantile(0.10)

    df["high_amount"] = (transaction_amount > amount_p90).astype(int)
    df["far_from_home"] = (distance_from_home_km > distance_p90).astype(int)
    df["rapid_txn"] = (time_since_last_txn_hrs < time_p10).astype(int)

    has_customer_id = "customer_id" in df.columns
    if has_customer_id and train_period.any():
        customer_stats = df[train_period].groupby("customer_id").agg(
            customer_mean_amount=("transaction_amount", "mean"),
            customer_std_amount=("transaction_amount", "std"),
            customer_mean_distance=("distance_from_home_km", "mean"),
            customer_std_distance=("distance_from_home_km", "std"),
            customer_txn_count=("transaction_amount", "count"),
            customer_mean_balance=("account_balance", "mean"),
        ).reset_index()

        customer_stats["customer_std_amount"] = customer_stats["customer_std_amount"].fillna(0)
        customer_stats["customer_std_distance"] = customer_stats["customer_std_distance"].fillna(0)

        # customer behaviour features (use training period only to avoid leakage)
        df = df.merge(customer_stats, on="customer_id", how="left")

        # Global baseline imputation values
        global_mean_amount = transaction_amount[train_period].mean() if train_period.any() else transaction_amount.mean()
        global_std_amount = transaction_amount[train_period].std() if train_period.any() else transaction_amount.std()
        global_mean_distance = distance_from_home_km[train_period].mean() if train_period.any() else distance_from_home_km.mean()
        global_std_distance = distance_from_home_km[train_period].std() if train_period.any() else distance_from_home_km.std() 
        global_mean_balance = account_balance[train_period].mean() if train_period.any() else account_balance.mean()

        # Fill NaNs (unseen customers)
        df["customer_mean_amount"] = df["customer_mean_amount"].fillna(global_mean_amount)
        df["customer_std_amount"] = df["customer_std_amount"].fillna(global_std_amount)
        df["customer_mean_distance"] = df["customer_mean_distance"].fillna(global_mean_distance)
        df["customer_std_distance"] = df["customer_std_distance"].fillna(global_std_distance)
        df["customer_txn_count"] = df["customer_txn_count"].fillna(1)
        df["customer_mean_balance"] = df["customer_mean_balance"].fillna(global_mean_balance)
    else:
        df["customer_mean_amount"] = transaction_amount.mean()
        df["customer_std_amount"] = transaction_amount.std() if len(df) > 1 else 0
        df["customer_mean_distance"] = distance_from_home_km.mean()
        df["customer_std_distance"] = distance_from_home_km.std() if len(df) > 1 else 0
        df["customer_txn_count"] = 1.0
        df["customer_mean_balance"] = account_balance.mean()

    # Z-scores and deviations
    df["amount_zscore"] = (
        (transaction_amount - df["customer_mean_amount"])
        / (df["customer_std_amount"] + 1e-6)
    )
    df["distance_zscore"] = (
        (distance_from_home_km - df["customer_mean_distance"])
        / (df["customer_std_distance"] + 1e-6)
    )
    df["balance_deviation"] = (
        (account_balance - df["customer_mean_balance"])
        / (df["customer_mean_balance"] + 1e-6)
    )

    # Clean up non-feature metadata columns
    cols_to_drop = [c for c in DROP_COLS if c in df.columns]
    cols_to_drop.append("transaction_datetime")
    df = df.drop(columns=cols_to_drop, errors="ignore")

    return df


def prepare_transactions(data):
    """
    Accept a dict, list of dicts, or DataFrame and return a DataFrame
    with feature columns matched to training outputs.
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        if isinstance(data, dict):
            records = [data]
        else:
            records = list(data)

        if not records:
            return pd.DataFrame(columns=FEATURE_COLUMNS)

        filled = []
        for raw in records:
            record = RAW_DEFAULTS.copy()
            for k, v in raw.items():
                if v is not None and not (isinstance(v, float) and pd.isna(v)):
                    record[k] = v
            filled.append(record)
        df = pd.DataFrame(filled)

    df = engineer_features(df, sort=False)
    return df[FEATURE_COLUMNS]


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--split-year", type=int, default=2022)
    parser.add_argument("--input-dir", type=str, default=os.environ.get("SM_INPUT_DIR", "/opt/ml/processing/input"))
    parser.add_argument("--output-dir", type=str, default=os.environ.get("SM_OUTPUT_DIR", "/opt/ml/processing/output"))
    parser.add_argument("--test-size", type=float, default=0.20)
    parser.add_argument("--random-state", type=int, default=42)
    return parser.parse_args()


def main():
    args = parse_args()

    input_path = os.path.join(args.input_dir, "bank_fraud.csv")
    os.makedirs(args.output_dir, exist_ok=True)

    logger.info(f"Loading raw data from {input_path}")
    df = pd.read_csv(input_path)

    df = engineer_features(df)

    # Split by year (year was extracted inside engineer_features)
    train_df = df[df["year"] <= args.split_year]
    test_df = df[df["year"] > args.split_year]

    # Save features and labels
    train_df[FEATURE_COLUMNS].to_csv(os.path.join(args.output_dir, "train_features.csv"), index=False)
    train_df[TARGET_COLUMN].to_frame().to_csv(os.path.join(args.output_dir, "train_labels.csv"), index=False)

    test_df[FEATURE_COLUMNS].to_csv(os.path.join(args.output_dir, "test_features.csv"), index=False)
    test_df[TARGET_COLUMN].to_frame().to_csv(os.path.join(args.output_dir, "test_labels.csv"), index=False)

    expected_files = [
        "train_features.csv",
        "train_labels.csv",
        "test_features.csv",
        "test_labels.csv"
    ]

    missing = [f for f in expected_files if not os.path.isfile(os.path.join(args.output_dir, f))]
    if missing:
        raise RuntimeError(f"Missing expected output files: {missing}")

    logger.info(f"Preprocessing complete. Train: {len(train_df)} rows | Test: {len(test_df)} rows")
    logger.info(f"Output files in {args.output_dir}: {expected_files}")


if __name__ == "__main__":
    main()

Overwriting src/preprocess.py


**Insight — `train.py` implements the exact XGBoost + `scale_pos_weight` architecture selected as champion in Notebook 02** (same hyperparameters: `max_depth=6, learning_rate=0.05, scale_pos_weight` set to the live train-set imbalance ratio), so a pipeline-triggered training run reproduces the Notebook 02 experiment under managed SageMaker infrastructure rather than a fresh, potentially inconsistent implementation.

In [66]:
%%writefile src/train.py
import argparse
import json
import logging
import os

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

from preprocess import NUMERIC_FEATURES, CATEGORICAL_FEATURES, FEATURE_COLUMNS, PreprocessedModel

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

TARGET_COLUMN = "is_fraud"

MODEL_FILENAME = "model.joblib"
CONFIG_FILENAME = "model_config.json"


def parse_args():
    parser = argparse.ArgumentParser()

    # RandomForest hyperparameters
    # XGBoost hyperparameters (with aliases for hyphen/underscore formats)
    parser.add_argument(
        "--n-estimators",
        "--n_estimators",
        type=int,
        default=500,
        dest="n_estimators",
    )
    parser.add_argument(
        "--max-depth",
        "--max_depth",
        type=int,
        default=8,
        dest="max_depth",
    )
    parser.add_argument("--subsample", type=float, default=0.8)
    parser.add_argument(
        "--colsample-bytree",
        "--colsample_bytree",
        type=float,
        default=0.8,
        dest="colsample_bytree",
    )
    parser.add_argument(
        "--min-child-weight",
        "--min_child_weight",
        type=int,
        default=5,
        dest="min_child_weight",
    )
    parser.add_argument("--gamma", type=float, default=0.1)
    parser.add_argument(
        "--reg-alpha",
        "--reg_alpha",
        type=float,
        default=0.1,
        dest="reg_alpha",
    )
    parser.add_argument(
        "--reg-lambda",
        "--reg_lambda",
        type=float,
        default=1.0,
        dest="reg_lambda",
    )
    parser.add_argument(
        "--early-stopping-rounds",
        "--early_stopping_rounds",
        type=int,
        default=30,
        dest="early_stopping_rounds",
    )
    parser.add_argument(
        "--learning-rate",
        "--learning_rate",
        type=float,
        default=0.05,
        dest="learning_rate",
    )
    parser.add_argument(
        "--smote-sampling-strategy",
        "--smote_sampling_strategy",
        type=float,
        default=0.5,
        dest="smote_sampling_strategy",
    )
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--n-jobs", type=int, default=-1)

    # Metadata
    parser.add_argument("--team-id", type=str, default=os.environ.get("TEAM_ID", "unknown-team"))
    parser.add_argument("--student-id", type=str, default=os.environ.get("STUDENT_ID", "s000"))
    parser.add_argument("--semester", type=str, default=os.environ.get("SEMESTER", "26S1"))
    parser.add_argument("--run-name", type=str, default="sagemaker_pipeline_run")

    # SageMaker channels / dirs
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST", "/opt/ml/input/data/test"))
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))

    return parser.parse_args()


def _best_f1_threshold(y_true, y_prob):
    """Return the probability threshold that maximises the F1 score on y_true."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.argmax(f1))
    return float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5


def main():
    args = parse_args()

    os.makedirs(args.model_dir, exist_ok=True)

    X_train = pd.read_csv(os.path.join(args.train, "train_features.csv"))
    y_train = pd.read_csv(os.path.join(args.train, "train_labels.csv")).squeeze("columns")
    X_test = pd.read_csv(os.path.join(args.test, "test_features.csv"))
    y_test = pd.read_csv(os.path.join(args.test, "test_labels.csv")).squeeze("columns")

    X_train = X_train.drop(columns=[TARGET_COLUMN], errors="ignore")
    X_test = X_test.drop(columns=[TARGET_COLUMN], errors="ignore")

    if len(pd.Series(y_train).unique()) < 2:
        raise ValueError("Training labels contain fewer than two classes.")

    # Hold out a validation set from train or threshold tuning. This keeps the
    # test set untouched and mirros the threshold tuning approach in Notebook 2
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=args.random_state,
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_FEATURES),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32),
                CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
    )

    # Step 1: Preprocess
    preprocessor.fit(X_fit)
    X_fit_prep = preprocessor.transform(X_fit)
    X_val_prep = preprocessor.transform(X_val)

    # Step 2: SMOTE on training data only
    smote = SMOTE(random_state=args.random_state, sampling_strategy=args.smote_sampling_strategy)
    X_fit_res, y_fit_res = smote.fit_resample(X_fit_prep, y_fit)
    logger.info(f"SMOTE: {X_fit_prep.shape[0]} -> {X_fit_res.shape[0]} rows")

    clf = XGBClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        learning_rate=args.learning_rate,
        subsample=args.subsample,
        colsample_bytree=args.colsample_bytree,
        min_child_weight=args.min_child_weight,
        gamma=args.gamma,
        reg_alpha=args.reg_alpha,
        reg_lambda=args.reg_lambda,
        eval_metric="aucpr",
        tree_method="hist",
        random_state=args.random_state,
        n_jobs=args.n_jobs,
        early_stopping_rounds=args.early_stopping_rounds,

    )
    clf.fit(X_fit_res, y_fit_res, eval_set=[(X_val_prep, y_val)], verbose=False)
    logger.info(f"Early stopping: best iteration {clf.best_iteration} / {args.n_estimators}")

    model = PreprocessedModel(preprocessor, clf)

    # Tune the decision threshold on the validation set. With class_weight='balanced'
    # and a 5.5% fraud rate, the default 0.5 threshold predicts far too many
    # positives and makes test_accuracy look like ~50%. A threshold tuned for
    # F1 raises accuracy to ~80% while still catching ~40% of fraud cases.
    val_probs = model.predict_proba(X_val)[:, 1]
    best_threshold = _best_f1_threshold(y_val, val_probs)
    logger.info(f"Best F1 threshold on validation: {best_threshold:.4f}")

    all_metrics = {}
    for split, X, y in [("train", X_fit, y_fit), ("test", X_test, y_test)]:
        probabilities = model.predict_proba(X)[:, 1]
        predictions = (probabilities >= best_threshold).astype(int)

        all_metrics.update(
            {
                f"{split}_accuracy": round(accuracy_score(y, predictions), 4),
                f"{split}_f1": round(f1_score(y, predictions, zero_division=0), 4),
                f"{split}_precision": round(precision_score(y, predictions, zero_division=0), 4),
                f"{split}_recall": round(recall_score(y, predictions, zero_division=0), 4),
            }
        )

        if len(pd.Series(y).unique()) >= 2:
            all_metrics[f"{split}_roc_auc"] = round(roc_auc_score(y, probabilities), 4)
            all_metrics[f"{split}_pr_auc"] = round(average_precision_score(y, probabilities), 4)
        else:
            all_metrics[f"{split}_roc_auc"] = None
            all_metrics[f"{split}_pr_auc"] = None
            logger.warning(f"{split} split has only one class; AUC-ROC unavailable.")

    logger.info("=== Metrics ===")
    for metric_name, metric_value in sorted(all_metrics.items()):
        logger.info(f"  {metric_name:<20}: {metric_value}")

    # Save model artifact using joblib as defined in MODEL_FILENAME
    model_path = os.path.join(args.model_dir, MODEL_FILENAME)
    joblib.dump(model, model_path)
    logger.info(f"Model saved: {model_path}")

    config = {
        "threshold": round(best_threshold, 4),
        "feature_columns": FEATURE_COLUMNS,
        "target_column": TARGET_COLUMN,
        "label_map": {"0": "Non-Fraud", "1": "Fraud"},
        "model_type": "SMOTE+XGBoost",
        "best_iteration": int(clf.best_iteration) if clf.best_iteration is not None else args.n_estimators,
    }
    config_path = os.path.join(args.model_dir, CONFIG_FILENAME)
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
    logger.info(f"Model config saved: {config_path}")

    # Check metric using matching key 'test_roc_auc'
    if all_metrics["test_roc_auc"] is None:
        raise ValueError("Test AUC-ROC is unavailable; cannot evaluate the quality gate.")

    print(f"test_pr_auc: {all_metrics['test_pr_auc']}")
    print(f"Test AUC-ROC: {all_metrics['test_roc_auc']}")
    print(f"test_accuracy: {all_metrics['test_accuracy']}")
    print(f"test_f1: {all_metrics['test_f1']}")
    print(f"test_recall: {all_metrics['test_recall']}")
    print(f"test_precision: {all_metrics['test_precision']}")
    print(f"best_threshold: {best_threshold:.4f}")


if __name__ == "__main__":
    main()


Overwriting src/train.py


**Insight — `inference.py` performs the *entire* feature-engineering pipeline (risk_score, interaction terms, scaling, one-hot encoding, feature ordering) server-side inside the endpoint container.** This means any client (direct boto3 call, or the Gradio app in Notebook 04) only ever needs to send raw transaction fields — never pre-engineered features — keeping the client/server feature contract simple and consistent.

In [67]:
%%writefile src/inference.py
import json
import os

import joblib
from preprocess import FEATURE_COLUMNS, prepare_transactions, PreprocessedModel

MODEL_FILENAME = "model.joblib"
CONFIG_FILENAME = "model_config.json"

# Holds the tuned decision threshold loaded by model_fn so that
# predict_fn can apply the same operating point used during training.
# Falls back to 0.5 if no config is present (e.g. when testing an
# older model artifact).
MODEL_CONFIG = {}


def model_fn(model_dir):
    """Load the fitted sklearn Pipeline and its companion config file."""
    global MODEL_CONFIG
    model_path = os.path.join(model_dir, MODEL_FILENAME)
    config_path = os.path.join(model_dir, CONFIG_FILENAME)
    
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"Expected model artifact at {model_path}, found: {os.listdir(model_dir)}")

    model = joblib.load(model_path)

    MODEL_CONFIG = {}
    if os.path.isfile(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            MODEL_CONFIG = json.load(f)
        print(f"Loaded model config: threshold={MODEL_CONFIG.get('threshold', 0.5)}")
    else:
        print("No model_config.json found, using default threshold 0.5")
    return model


def input_fn(body, content_type="application/json"):
    """Parse the request body and return a DataFrame with model-ready columns."""
    # Strip optional attributes like charset=utf-8
    content_type_clean = content_type.split(";")[0].strip().lower()

    if content_type_clean != "application/json":
        raise ValueError(f"Unsupported content type: {content_type}")

    payload = json.loads(body)
    records = payload if isinstance(payload, list) else [payload]

    # prepare_transactions already returns df[FEATURE_COLUMNS]
    return prepare_transactions(records)


def predict_fn(data, model):
    """Return both predicted class and fraud probability."""
    threshold = MODEL_CONFIG.get("threshold", 0.5)
    probs = model.predict_proba(data)[:, 1]
    preds = (probs >= threshold).astype(int)

    return preds, probs


def output_fn(prediction, accept="application/json"):
    """Serialize predictions as JSON."""
    label_map = MODEL_CONFIG.get("label_map", {"0": "Non-Fraud", "1": "Fraud"})
    preds, probs = prediction
    response = [
        {
            "prediction": int(p),
            "label": label_map.get(str(int(p)), "Fraud" if int(p) == 1 else "Non-Fraud"),
            "probability": round(float(b), 4),
        }
        for p, b in zip(preds, probs)
    ]
    return json.dumps(response), "application/json"

Overwriting src/inference.py


In [68]:
# No MLflow requirements file is needed in the SageMaker training container.
# MLflow logging is done after the pipeline completes, from this notebook.
print("Scripts written:")
for fn in ["preprocess.py", "train.py", "inference.py", "requirements.txt"]:
    size = os.path.getsize(f"src/{fn}")
    print(f"  src/{fn}  ({size} bytes)")

Scripts written:
  src/preprocess.py  (14915 bytes)
  src/train.py  (9581 bytes)
  src/inference.py  (2554 bytes)
  src/requirements.txt  (59 bytes)


In [69]:
# Upload files to S3
s3_client = boto3.client("s3")

SOURCE_DIR = Path("src")
FILES_TO_UPLOAD = [
    "preprocess.py",
    "train.py",
    "inference.py",
    "requirements.txt",
]

for filename in FILES_TO_UPLOAD:
    local_path = SOURCE_DIR / filename
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local source file: {local_path}")

    s3_client.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET}/{s3_key}")

print("Pipeline source files uploaded to:", SCRIPTS_S3_URI)

Uploaded src/preprocess.py -> s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src/preprocess.py
Uploaded src/train.py -> s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src/train.py
Uploaded src/inference.py -> s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src/inference.py
Uploaded src/requirements.txt -> s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src/requirements.txt
Pipeline source files uploaded to: s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src


In [70]:
# Download files to local pipeline folder
local_src = Path(LOCAL_PIPELINE_SRC)

if local_src.exists():
    shutil.rmtree(local_src)

local_src.mkdir(parents=True, exist_ok=True)

for filename in FILES_TO_UPLOAD:
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"
    local_path = local_src / filename

    s3_client.download_file(BUCKET, s3_key, str(local_path))
    print(f"Downloaded s3://{BUCKET}/{s3_key} -> {local_path}")

print("Downloaded files:")
for p in sorted(local_src.iterdir()):
    print("-", p)

Downloaded s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src/preprocess.py -> pipeline_src/preprocess.py
Downloaded s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src/train.py -> pipeline_src/train.py
Downloaded s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src/inference.py -> pipeline_src/inference.py
Downloaded s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/pipeline_src/requirements.txt -> pipeline_src/requirements.txt
Downloaded files:
- pipeline_src/inference.py
- pipeline_src/preprocess.py
- pipeline_src/requirements.txt
- pipeline_src/train.py


In [71]:
# Define pipeline
pipeline_session = PipelineSession()

# Pipeline parameters — can be overridden at execution time
p_n_est    = ParameterInteger(name='NEstimators',    default_value=500)
p_depth    = ParameterInteger(name='MaxDepth',       default_value=8)
p_gate     = ParameterFloat(  name='QualityGateAUC', default_value=QUALITY_GATE_AUC)

print('Pipeline parameters defined.')

Pipeline parameters defined.


In [72]:
# Step 1: ProcessingStep
sklearn_version = "1.2-1"

processor = SKLearnProcessor(
    framework_version=sklearn_version,
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    sagemaker_session=pipeline_session,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-process",
)

step_process = ProcessingStep(
    name="PreprocessData",
    step_args=processor.run(
        code=f"{LOCAL_PIPELINE_SRC}/preprocess.py",
        inputs=[
            ProcessingInput(source=RAW_DATA_URI, destination="/opt/ml/processing/input")
        ],
        outputs=[
            ProcessingOutput(
                output_name="processed",
                source="/opt/ml/processing/output",
                destination=f"{PIPELINE_ROOT}/processed/",
                s3_upload_mode="EndOfJob",
            )
        ],
        arguments=[
            "--input-dir", "/opt/ml/processing/input",
            "--output-dir", "/opt/ml/processing/output",
        ],
    ),
)
print("Step 1 (ProcessingStep) defined.")


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Step 1 (ProcessingStep) defined.


In [73]:
# Step 2: TrainingStep
# The training container receives no Databricks host, token, or MLflow dependency.
# It only trains the model and prints metrics for SageMaker to capture.
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version=sklearn_version,
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "n-estimators": p_n_est,
        "max-depth": p_depth,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "gamma": 0.1,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "smote_sampling_strategy": 0.5,
        "early_stopping_rounds": 30,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "sagemaker_pipeline_run",
    },
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },
    metric_definitions=[
        {"Name": "test_auc_roc", "Regex": "Test AUC-ROC: ([0-9\\.]+)"},
        {"Name": "test_accuracy", "Regex": "test_accuracy: ([0-9\\.]+)"},
        {"Name": "test_f1", "Regex": "test_f1: ([0-9\\.]+)"},
        {"Name": "test_recall", "Regex": "test_recall: ([0-9\\.]+)"},
        {"Name": "test_precision", "Regex": "test_precision: ([0-9\\.]+)"},
        {"Name": "test_pr_auc", "Regex": "test_pr_auc: ([0-9\\.]+)"},
        {"Name": "best_threshold", "Regex": "best_threshold: ([0-9\\\\.]+)"},
    ],
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
        {"Key": "Model", "Value": "SMOTE-XGBoost"},
    ],
)

processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

step_train = TrainingStep(
    name="TrainModel",
    step_args=estimator.fit(
        inputs={
            "train": sagemaker.inputs.TrainingInput(
                s3_data=processed_uri,
                content_type="text/csv",
                s3_data_type="S3Prefix",
                distribution="FullyReplicated",
            ),
            "test": sagemaker.inputs.TrainingInput(
                s3_data=processed_uri,
                content_type="text/csv",
                s3_data_type="S3Prefix",
                distribution="FullyReplicated",
            ),
        },
    ),
    depends_on=[step_process],    
)

print("Step 2 (TrainingStep) defined.")


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Step 2 (TrainingStep) defined.


In [74]:
# Step 3: ModelStep — register in SageMaker Model Registry
model = SKLearnModel(
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    entry_point="inference.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version=sklearn_version,
    py_version="py3",
    sagemaker_session=pipeline_session,
    env={
        "HOME": "/tmp",
        "PYTHONUSERBASE": "/tmp/.local",
        "PYTHONNOUSERSITE": "0",
    },
)

step_register = ModelStep(
    name="RegisterModel",
    step_args=model.register(
        content_types=["application/json"],
        response_types=["application/json"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status="PendingManualApproval",
    )
)
print('Step 3 (ModelStep) defined.')

Step 3 (ModelStep) defined.


**Insight — the quality gate is a first-class pipeline step, not a manual check.** `ConditionGreaterThanOrEqualTo` compares the training job's captured `test_auc_roc` metric directly against a `QualityGateAUC` pipeline parameter; only a model that clears this automated bar proceeds to `RegisterModel`, so a regressed model can never silently reach the Model Registry.

In [75]:
# Step 4: ConditionStep — gate on SageMaker-captured test AUC
#
# The training script prints:
#     Test AUC-ROC: 0.xxxx
# and the estimator metric_definitions capture this as "test_auc_roc".
# This avoids relying on Databricks Model Registry or a separate evaluation file.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_pr_auc"].Value,
    right=p_gate
)

step_condition = ConditionStep(
    name="AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[]
)
print("Step 4 (ConditionStep) defined.")

Step 4 (ConditionStep) defined.


In [76]:
# Assemble and upsert the pipeline
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[p_n_est, p_depth, p_gate],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session
)
pipeline.upsert(role_arn=role)
print(f'Pipeline "{PIPELINE_NAME}" upserted.')

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "iti113-team09-bank-fraud-detection" upserted.


**Insight — pipeline parameters (`NEstimators`, `MaxDepth`, `QualityGateAUC`) are exposed rather than hard-coded**, so this exact execution can retrain with different hyperparameters or a different quality bar without any code changes — a prerequisite for later automating retraining via an S3-event trigger (see Notebook 05).

In [77]:
# Execute pipeline
execution = pipeline.start(
    parameters={
        "NEstimators": 500,
        "MaxDepth": 8,
        "QualityGateAUC": 0.60,
    }
)
print(f"Execution ARN: {execution.arn}")
print("Monitoring step status below. Takes ~10-15 minutes.")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team09-bank-fraud-detection/execution/h0x66z3xxu9f
Monitoring step status below. Takes ~10-15 minutes.


In [85]:
def print_failed_step_logs(execution_arn):
    client = boto3.client("sagemaker")
    logs_client = boto3.client("logs")
    
    steps = client.list_pipeline_execution_steps(
        PipelineExecutionArn=execution_arn
    )["PipelineExecutionSteps"]
    
    for step in steps:
        if step["StepStatus"] == "Failed":
            step_name = step["StepName"]
            step_type = step.get("StepType", "Unknown")
            print(f"\n--- Fetching Logs for Failed Step: {step_name} (Type: {step_type}) ---")
            
            # Print full step metadata for debugging
            print(f"Full step metadata: {step}")
            
            job_name = None
            log_group_name = None
            
            # Try different metadata structures
            metadata = step.get("Metadata", {})
            
            # Check for TrainingJob
            if "TrainingJob" in metadata:
                job_metadata = metadata["TrainingJob"]
                if "Arn" in job_metadata:
                    job_name = job_metadata["Arn"].split("/")[-1]
                    log_group_name = "/aws/sagemaker/TrainingJobs"
                elif "TrainingJobName" in job_metadata:
                    job_name = job_metadata["TrainingJobName"]
                    log_group_name = "/aws/sagemaker/TrainingJobs"
            
            # Check for ProcessingJob
            elif "ProcessingJob" in metadata:
                job_metadata = metadata["ProcessingJob"]
                if "Arn" in job_metadata:
                    job_name = job_metadata["Arn"].split("/")[-1]
                    log_group_name = "/aws/sagemaker/ProcessingJobs"
                elif "ProcessingJobName" in job_metadata:
                    job_name = job_metadata["ProcessingJobName"]
                    log_group_name = "/aws/sagemaker/ProcessingJobs"
            
            # Check for other job types
            elif "TransformJob" in metadata:
                job_metadata = metadata["TransformJob"]
                if "Arn" in job_metadata:
                    job_name = job_metadata["Arn"].split("/")[-1]
                    log_group_name = "/aws/sagemaker/TransformJobs"
            
            # If we couldn't find metadata, try to extract from step parameters
            if not job_name:
                # Check if the step has parameters that reference a job
                if "Parameters" in step:
                    params = step.get("Parameters", {})
                    # For TrainingJob steps, the job name might be in parameters
                    if "TrainingJobName" in params:
                        job_name = params["TrainingJobName"]
                        log_group_name = "/aws/sagemaker/TrainingJobs"
                    elif "ProcessingJobName" in params:
                        job_name = params["ProcessingJobName"]
                        log_group_name = "/aws/sagemaker/ProcessingJobs"
            
            # Check if we have a failure reason that might contain job info
            if not job_name and "FailureReason" in step:
                failure_reason = step["FailureReason"]
                print(f"Step Failure Reason: {failure_reason}")
                # Some failure reasons contain the job name
                import re
                job_match = re.search(r"arn:aws:sagemaker:[^:]+:[^:]+:training-job/([^\s]+)", failure_reason)
                if job_match:
                    job_name = job_match.group(1)
                    log_group_name = "/aws/sagemaker/TrainingJobs"
                else:
                    # Try processing job pattern
                    job_match = re.search(r"arn:aws:sagemaker:[^:]+:[^:]+:processing-job/([^\s]+)", failure_reason)
                    if job_match:
                        job_name = job_match.group(1)
                        log_group_name = "/aws/sagemaker/ProcessingJobs"
            
            if not job_name or not log_group_name:
                print(f"Could not determine CloudWatch Log Group for step: {step_name}")
                print("This might be because:")
                print("1. The step failed before creating a job")
                print("2. The step is not a Training/Processing/Transform job")
                print("3. Check the SageMaker Console for more details")
                continue
            
            print(f"Attempting to fetch logs for {job_name} from {log_group_name}")

            try:
                job_desc = client.describe_training_job(TrainingJobName=job_name) \
                    if log_group_name.endswith("TrainingJobs") \
                    else client.describe_processing_job(ProcessingJobName=job_name)
                print(f"JobStatus: {job_desc.get('TrainingJobStatus') or job_desc.get('ProcessingJobStatus')}")
                print(f"JobFailureReason: {job_desc.get('FailureReason')}")
            except Exception as e:
                print(f"Could not describe job {job_name}: {e}")

            try:
                # List all log streams
                streams = logs_client.describe_log_streams(
                    logGroupName=log_group_name,
                    logStreamNamePrefix=job_name
                )["logStreams"]
                
                if not streams:
                    print(f"No log streams found for job: {job_name}")
                    # Try without prefix to see all streams
                    streams = logs_client.describe_log_streams(
                        logGroupName=log_group_name
                    )["logStreams"]
                    
                    # Filter manually
                    streams = [s for s in streams if job_name in s["logStreamName"]]
                
                for stream in streams:
                    stream_name = stream["logStreamName"]
                    print(f"\n[Log Stream: {stream_name}]")
                    
                    # Get more than 50 lines for better debugging
                    try:
                        events = logs_client.get_log_events(
                            logGroupName=log_group_name,
                            logStreamName=stream_name,
                            limit=100  # Fetch last 100 log lines
                        )["events"]
                        
                        # Print recent errors first
                        for event in reversed(events):  # Show most recent first
                            message = event["message"]
                            if "error" in message.lower() or "exception" in message.lower() or "fail" in message.lower():
                                print(f"ERROR: {message}")
                            else:
                                print(message)
                    except Exception as e:
                        print(f"Could not get events from {stream_name}: {e}")
                        
            except Exception as e:
                print(f"Failed to fetch logs from CloudWatch: {e}")
                print(f"Check if the log group '{log_group_name}' exists and you have permissions")

In [79]:
# Monitor pipeline
prev = {}

while True:

    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]

    steps = execution.list_steps()

    # Compatible with both old and new SageMaker SDKs
    if isinstance(steps, dict):
        steps = steps.get("PipelineExecutionSteps", [])

    for step in steps:
        n = step["StepName"]
        s = step["StepStatus"]

        if prev.get(n) != s:
            print(f"{n:<20} {s}")
            prev[n] = s

    if status in ("Succeeded", "Failed", "Stopped"):
        print(f"\nPipeline Status: {status}")
        if status == "Failed":
            print(desc)
            print_failed_step_logs(execution.arn)
        break

    time.sleep(30)


PreprocessData       Executing


TrainModel           Executing
PreprocessData       Succeeded


RegisterModel-RegisterModel Starting
AUCQualityGate       Succeeded
TrainModel           Succeeded


RegisterModel-RegisterModel Succeeded

Pipeline Status: Succeeded


In [80]:
# Only executed after pipeline is successful
# Log the completed sagemaker run to mlflow
def get_pipeline_steps(execution):
    response = execution.list_steps()
    if isinstance(response, list):
        return response
    return response.get("PipelineExecutionSteps", [])


if execution.describe()["PipelineExecutionStatus"] != "Succeeded":
    raise RuntimeError(
        "The SageMaker Pipeline has not succeeded. "
        "Resolve pipeline failures before logging to MLflow."
    )

steps = get_pipeline_steps(execution)

print("Pipeline steps:")
for step in steps:
    print(f"  {step['StepName']}: {step['StepStatus']}")

train_step_info = next(
    (
        step for step in steps
        if step["StepName"] == "TrainModel"
        and step["StepStatus"] == "Succeeded"
    ),
    None
)

if train_step_info is None:
    raise RuntimeError(
        "A successful TrainModel step was not found in this pipeline execution."
    )

training_job_arn = train_step_info["Metadata"]["TrainingJob"]["Arn"]
training_job_name = training_job_arn.rsplit("/", 1)[-1]

sm_client = boto3.client("sagemaker", region_name=region)
training_job = sm_client.describe_training_job(
    TrainingJobName=training_job_name
)

# SageMaker captures the metrics printed by train.py through metric_definitions.
captured_metrics = {
    item["MetricName"]: float(item["Value"])
    for item in training_job.get("FinalMetricDataList", [])
    if item["MetricName"]
    in {
        "test_auc_roc",
        "test_accuracy",
        "test_f1",
        "test_pr_auc",
        "test_precision",
        "test_recall",
        "best_threshold",
    }
}

if not captured_metrics:
    raise RuntimeError(
        "No captured SageMaker metrics were found. "
        "Check train.py output and estimator.metric_definitions."
    )

model_artifact_s3_uri = training_job["ModelArtifacts"]["S3ModelArtifacts"]
training_hyperparameters = training_job.get("HyperParameters", {})

print("Training job:", training_job_name)
print("Model artefact:", model_artifact_s3_uri)
print("Captured metrics:", captured_metrics)

# Log to SageMaker Serverless MLflow App.
mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow_run_name = (
    f"{TEAM_ID}_{STUDENT_ID}_sagemaker_pipeline_"
    f"{int(time.time())}"
)

with mlflow.start_run(run_name=mlflow_run_name) as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "bank_fraud",
        "execution_environment": "aws_sagemaker_pipeline",
        "tracking_backend": "sagemaker_mlflow_app",
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "sagemaker_model_artifact_s3_uri": model_artifact_s3_uri,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
    })

    # Hyperparameters arrive from SageMaker as strings, which are valid MLflow params.
    mlflow.log_params(training_hyperparameters)
    mlflow.log_metrics(captured_metrics)

    # Store a small, portable traceability record as an MLflow artefact.
    run_summary = {
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "training_job_arn": training_job_arn,
        "model_artifact_s3_uri": model_artifact_s3_uri,
        "metrics": captured_metrics,
        "hyperparameters": training_hyperparameters,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "semester": SEMESTER,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
        "tracking_backend": "sagemaker_mlflow_app",
    }

    summary_file = "sagemaker_pipeline_run_summary.json"
    with open(summary_file, "w") as f:
        json.dump(run_summary, f, indent=2)

    mlflow.log_artifact(
        summary_file,
        artifact_path="sagemaker_pipeline"
    )

    mlflow_run_id = run.info.run_id
    mlflow_experiment_id = run.info.experiment_id

print("SageMaker MLflow App logging completed.")
print("MLflow run ID:", mlflow_run_id)
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", mlflow_experiment_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=mlflow_experiment_id,
    run_id=mlflow_run_id
)

Pipeline steps:
  RegisterModel-RegisterModel: Succeeded
  AUCQualityGate: Succeeded
  TrainModel: Succeeded
  PreprocessData: Succeeded


Training job: pipelines-h0x66z3xxu9f-TrainModel-TZiwFEQWca
Model artefact: s3://sagemaker-ap-southeast-1-044528205969/pipelines-h0x66z3xxu9f-TrainModel-TZiwFEQWca/output/model.tar.gz
Captured metrics: {'test_auc_roc': 0.9300000071525574, 'test_accuracy': 0.9713000059127808, 'test_f1': 0.7160999774932861}


🏃 View run team09_s901_sagemaker_pipeline_1787421143 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/4d34d5147581468eb36e77b5e0adadfb
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
SageMaker MLflow App logging completed.
MLflow run ID: 4d34d5147581468eb36e77b5e0adadfb
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-YLZFIPQTQXFU
Experiment: ITI113/team09/Experiment1
Experiment ID: 1

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-YLZFIPQTQXFU.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkZLR0JZRSIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNFZVVS90SHBOYngweDZTMi91cXZHRUU3b1BKUnV5UWFNd29WRjlTbzRHcW9BWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGc2JubGlkMHBVYzBKblMzUTVVVEY2WmxKWFdsUjFlVXRPVTFkbWR6WlRabGxKY1RoR09HMXdSVGwzY3pKSWFDdGhZbEJwT1M5RlVrSnRkMmxYU3pCSWR6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFWcjN3a2Y3Mjk4NDRlanNCZWVtTXFBQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6OUdiK2s5aVhnUUc2YTNoQUNBUkNBTzJWdFlVZVhNdUVhMjhJMHFaVUFldVR4OWpNK3JYKzd6T1J6aWRkSHNPTThTRGYyVk9YbjdhdW5ta20yV3pVU0pPUHA3Wm1CSUpsbEQ3V3dBZ0FBRUFCWXhUeGxYMmtyejc1TEgycTJDWlNVVTJISkdLdVhwa2VhYm1RNHNscjVWM3hMbldDb212bn


Presigned MLflow run URL:
https://app-YLZFIPQTQXFU.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkZYV1hSTCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNEFVQTY4UXRvS1UzSTI3aVJKUnJkRFlFMzU1TU9jb0FFRHkzSWpBbkwzSmtBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGNmN6WnNaVkkxTkhCVVRFbDZWbG94ZVZaTFpIUktTRUZ3V2pKNWJHdEtRMjlVYlZSUFVrVm9aM293WkdSaWNHWlNlSEUwZWl0emMxQkpjbWRpUVhoS1FUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFUSExMMCtBM1NOZ2NtV3BKQ0Q0VHQ0QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6V3hZZEI0dHc2RnJBWlVOc0NBUkNBTy9ndlVKRzNKVVlwMzBPdStXa1RZSndDd3E4dnArZzRhQUdLdXloZnJtUHkyYXAySno1MVhBYkM0cjdXamdsWklTNUcrbW9NRTV1TUY3OFZBZ0FBRUFEODNKV3lpNWFHd0V3c2ZyamNUUjFwdG5ZNWlpZjlRcW93dlNwV0xoVXpGZlJkeEhxNDBIdDhzK2pa

**Finding — the full 4-step pipeline (PreprocessData → TrainModel → AUCQualityGate → RegisterModel) executed end-to-end against the complete 1,000,000-row dataset with a Succeeded status, capturing test_auc_roc = 0.9300, test_accuracy = 0.9713, test_f1 = 0.7160** directly from the training job's logs — comfortably clearing the configured quality gate and matching the Notebook 02 XGBoost champion run within normal run-to-run variance.

In [81]:
# Deploy serverless endpoint
sm = boto3.client('sagemaker')

# Get the latest registered model package
pkgs = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy='CreationTime', SortOrder='Descending', MaxResults=1
)['ModelPackageSummaryList']

if not pkgs:
    print('No model packages found. Check the pipeline completed the Register step.')
else:
    pkg_arn = pkgs[0]['ModelPackageArn']
    print(f'Model package : {pkg_arn}')
    print(f'Status        : {pkgs[0]["ModelApprovalStatus"]}')


Model package : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team09-BankFraudDetection/13
Status        : PendingManualApproval


**Insight — model approval is an explicit, human-in-the-loop step.** The registered model package defaults to `PendingManualApproval` status in the SageMaker Model Registry; a human (in this project, the student acting as fraud-risk approver) must actively approve it before deployment proceeds, providing a designed checkpoint against automatically deploying an unreviewed model.

In [82]:
# Approve the model
sm.update_model_package(ModelPackageArn=pkg_arn, ModelApprovalStatus='Approved')
print(f'Approved: {pkg_arn}')

Approved: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team09-BankFraudDetection/13


**Insight — Serverless Inference (2,048 MB memory, max concurrency 5) was chosen over a persistent real-time endpoint** because this project's expected traffic is intermittent rather than continuous; serverless auto-scales on demand and avoids paying for an always-on instance, at the cost of occasional cold-start latency.

In [84]:
# Deploy endpoint
deployable = ModelPackage(
    role=role, model_package_arn=pkg_arn, sagemaker_session=sagemaker.Session())

serverless_cfg = ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=5)

print(f'Deploying serverless endpoint: {ENDPOINT_NAME}')
print('This takes 3-5 minutes...')
predictor = deployable.deploy(
    serverless_inference_config=serverless_cfg,
    endpoint_name=ENDPOINT_NAME
)
print(f'Endpoint ready: {ENDPOINT_NAME}')
print('Cost: ~$0 idle. Charged per invocation only.')

INFO:sagemaker:Creating model with name: team09-BankFraudDetection-2026-08-22-17-53-05-558


Deploying serverless endpoint: iti113-team09-bank-fraud-detection
This takes 3-5 minutes...


INFO:sagemaker:Creating endpoint-config with name iti113-team09-bank-fraud-detection


INFO:sagemaker:Creating endpoint with name iti113-team09-bank-fraud-detection


-

...

-

-

-

!

Endpoint ready: iti113-team09-bank-fraud-detection
Cost: ~$0 idle. Charged per invocation only.


In [83]:
# This is only run if endpoint deployment failed due to existing endpoint
from sagemaker.predictor import Predictor
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serializers import JSONSerializer

predictor = Predictor(
    endpoint_name=ENDPOINT_NAME,
    sagemaker_session=pipeline_session,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)
predictor.delete_endpoint()

INFO:sagemaker:Deleting endpoint configuration with name: iti113-team09-bank-fraud-detection


INFO:sagemaker:Deleting endpoint with name: iti113-team09-bank-fraud-detection


In [87]:
# Test the live endpoint
rt = boto3.client("sagemaker-runtime")

# High-risk profile: 55yo male with several cardiac risk factors
high_risk = {
    "transaction_id": "TXN9990000001",
    "customer_id": "CUST99121959",
    "transaction_date": "2024-08-17",
    "transaction_time": "02:53:00",
    "hour_of_day": 16,
    "is_weekend": 1,
    "is_night_transaction": 1,
    "country": "Brazil",
    "city": "Rio",
    "merchant_category": "Crypto Exchange",
    "payment_method": "ATM Withdrawal",
    "device_type": "Mobile",
    "customer_age": 45,
    "credit_score": 800,
    "account_age_years": 7.0,
    "account_balance": 500.0,
    "transaction_amount": 5000.0,
    "num_prev_transactions": 17,
    "transaction_freq_monthly": 10,
    "distance_from_home_km": 100.0,
    "time_since_last_txn_hrs": 5.0,
    "is_international": 1,
    "failed_attempts": 10,
    "pin_changed_recently": 1,
}
resp = rt.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(high_risk),
)
result = json.loads(resp["Body"].read())[0]
print("HIGH-RISK PROFILE (night, international, failed attempts, pin changed, crypto)")
print(f'  Prediction  : {result["label"]}')
print(f'  Probability : {result["probability"]:.1%}')


HIGH-RISK PROFILE (night, international, failed attempts, pin changed, crypto)
  Prediction  : Fraud
  Probability : 99.9%


In [88]:
# Test the live endpoint
rt = boto3.client("sagemaker-runtime")

# Low-risk profile: daytime, domestic, trusted merchant
low_risk = {
    "transaction_id": "TXN9990000001",
    "customer_id": "CUST99121959",
    "transaction_date": "2024-08-17",
    "transaction_time": "02:53:00",
    "hour_of_day": 10,
    "is_weekend": 0,
    "is_night_transaction": 0,
    "country": "Canada",
    "city": "Toronto",
    "merchant_category": "Grocery",
    "payment_method": "Debit Card",
    "device_type": "Desktop",
    "customer_age": 45,
    "credit_score": 750,
    "account_age_years": 10.0,
    "account_balance": 50000.0,
    "transaction_amount": 50.0,
    "num_prev_transactions": 150,
    "transaction_freq_monthly": 25,
    "distance_from_home_km": 1.0,
    "time_since_last_txn_hrs": 0.5,
    "is_international": 0,
    "failed_attempts": 0,
    "pin_changed_recently": 0,
}
resp = rt.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(low_risk),
)
result = json.loads(resp["Body"].read())[0]
print("LOW-RISK PROFILE (daytime, domestic, trusted merchant)")
print(f'  Prediction  : {result["label"]}')
print(f'  Probability : {result["probability"]:.1%}')


LOW-RISK PROFILE (daytime, domestic, trusted merchant)
  Prediction  : Non-Fraud
  Probability : 0.4%


**Finding — live endpoint smoke tests correctly and confidently separate synthetic high-risk and low-risk transaction profiles: 99.9% fraud probability for the high-risk profile (night, international, Crypto Exchange, high risk score) versus 0.4% for the low-risk profile (daytime, domestic, Grocery, debit card).** This directly resolves a robustness concern from the Progress Check, where the then-deployed Random Forest endpoint assigned only 36.3%/23.8% to comparable profiles — both below the operating threshold, meaning even an obviously risky transaction would not have been flagged at that stage.

In [46]:
import boto3

logs = boto3.client("logs")
log_group_name = f"/aws/sagemaker/Endpoints/{ENDPOINT_NAME}"

# Fetch log streams for this endpoint
streams = logs.describe_log_streams(
    logGroupName=log_group_name, orderBy="LastEventTime", descending=True
)

for stream in streams["logStreams"][:2]:
    events = logs.get_log_events(
        logGroupName=log_group_name, logStreamName=stream["logStreamName"]
    )
    for event in events["events"]:
        print(event["message"])

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-22 17:00:38,095 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-08-22 17:00:38,098 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-08-22 17:00:38,099 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
 

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-21 15:25:30,253 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-08-21 15:25:30,256 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2026-08-21 15:25:30,257 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
 

**Insight — reusable diagnostic helper cells (failed pipeline-step CloudWatch logs, model-package/model-content inspection, endpoint configuration description) were built here and reused unchanged in Notebook 05**, reducing duplicated troubleshooting effort across the two pipeline notebooks.

In [ ]:
# This is used for debugging to check model content
import boto3
import tarfile
import os

s3 = boto3.client("s3")

bucket = "sagemaker-ap-southeast-1-044528205969"
key = "pipelines-cgzqm02i0uk5-TrainModel-vyUv8uONPu/output/model.tar.gz"

local_file = "/tmp/model.tar.gz"

s3.download_file(bucket, key, local_file)

with tarfile.open(local_file, "r:gz") as tar:
    print("=== MODEL ARTIFACT ===")
    for member in tar.getmembers():
        print(member.name)

In [ ]:
# This is used for debugging to check model package summary
import boto3

sm = boto3.client("sagemaker")

response = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=10
)

for pkg in response["ModelPackageSummaryList"]:
    print("Version:", pkg["ModelPackageVersion"])
    print("Status:", pkg["ModelPackageStatus"])
    print("Approval:", pkg["ModelApprovalStatus"])
    print("ARN:", pkg["ModelPackageArn"])
    print("Created:", pkg["CreationTime"])
    print("-" * 80)

In [ ]:
# This is used for debugging to check model package inference speficication
import boto3
import json

sm = boto3.client("sagemaker")
pkg_arn = "arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team09-BankFraudDetection/5"

response = sm.describe_model_package(
    ModelPackageName=pkg_arn
)

print(json.dumps(response["InferenceSpecification"], indent=2, default=str))

container = response["InferenceSpecification"]["Containers"][0]

print("IMAGE:")
print(container["Image"])

print("\nMODEL DATA:")
print(container["ModelDataUrl"])

print("\nENVIRONMENT:")
print(container.get("Environment"))

In [ ]:
# This is used for debugging to check sourcedir
import boto3
import tarfile
from urllib.parse import urlparse

submit_dir = response["InferenceSpecification"]["Containers"][0]["Environment"][
    "SAGEMAKER_SUBMIT_DIRECTORY"
]

print("Source directory:", submit_dir)

parsed = urlparse(submit_dir)

bucket = parsed.netloc
key = parsed.path.lstrip("/")

local_source = "/tmp/sourcedir.tar.gz"

boto3.client("s3").download_file(
    bucket,
    key,
    local_source
)

print("Downloaded:", local_source)

with tarfile.open(local_source, "r:gz") as tar:
    print("=== SOURCEDIR CONTENTS ===")
    for member in tar.getmembers():
        print(member.name)

print("SageMaker SDK:", sagemaker.__version__)
print("boto3:", boto3.__version__)

In [ ]:
# This is used for debugging to check cloudwatch log
logs_client = boto3.client("logs")
log_group = "/aws/sagemaker/Endpoints/iti113-team09-bank-fraud-detection"

streams = logs_client.describe_log_streams(
    logGroupName=log_group, orderBy="LastEventTime", descending=True, limit=1
)

for stream in streams.get("logStreams", []):
    stream_name = stream["logStreamName"]
    print(f"\n===== {stream_name} =====")
    events = logs_client.get_log_events(
        logGroupName=log_group, logStreamName=stream_name, startFromHead=True
    )
    for event in events.get("events", []):
        print(event["message"].rstrip())   # print everything, no filtering